# Module 11 · 4：音訊下游 — Whisper ASR / wav2vec2 分類

> 音訊資料（M09 音訊小節）整理好後的兩條主路線：
> **(A) Whisper 語音辨識 (ASR)**、**(B) wav2vec2 音訊分類**。

## 學習目標
- 用 `pipeline` 跑 **Whisper 語音辨識** 推論（最快上手）。
- 認識 **wav2vec2 微調**做音訊分類的資料格式與設定。

In [ ]:
import numpy as np
try:
    import torch
    HAS = True
except Exception:
    HAS = False
    print("需 `uv sync --extra multimodal --extra train`。說明仍可閱讀。")

## A. Whisper 語音辨識（推論）

最簡單的用法是 `transformers.pipeline`。輸入波形（16kHz），輸出轉錄文字。
首次執行會下載 `whisper-tiny`（最小版，CPU 可跑）。

In [ ]:
if HAS:
    try:
        import librosa
        from transformers import pipeline
        sr = 16000
        # 真實英語語音（librosa 內建 LibriSpeech 片段）
        wave, sr = librosa.load(librosa.example("libri1"), sr=sr, duration=10.0)

        asr = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")
        result = asr({"array": wave, "sampling_rate": sr})
        print("Whisper 轉錄結果:", result["text"])
        print("（真實語音 → Whisper 輸出實際的英文轉錄文字。）")
    except Exception as e:
        print("（未能下載 Whisper，略過）", e)
        print("流程：波形(16k) → Whisper pipeline → 轉錄文字。微調 ASR 用 Seq2SeqTrainer。")

## B. wav2vec2 音訊分類（微調）

資料格式：波形 `(N, samples)` → `input_values` + 標籤 `(N,)`。
用 `AutoModelForAudioClassification` + `Trainer`（與文字/影像同一套訓練 API）。

In [ ]:
if HAS:
    print("""微調設定骨架（概念）：

    from transformers import (AutoFeatureExtractor,
        AutoModelForAudioClassification, TrainingArguments, Trainer)

    fe = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")
    model = AutoModelForAudioClassification.from_pretrained(
        "facebook/wav2vec2-base", num_labels=NUM_CLASSES)

    # dataset: 每筆 = 16kHz 波形 + 標籤；用 fe(...) 轉成 input_values
    args = TrainingArguments(output_dir="...", num_train_epochs=3,
                             per_device_train_batch_size=8, learning_rate=3e-5)
    Trainer(model=model, args=args, train_dataset=ds).train()
    """)
    print("→ 與 DistilBERT/ViT 微調是同一套 Trainer 流程，只是換了模型與前處理。")

## 小結

| 路線 | 任務 | 模型 | 資料 |
|:--|:--|:--|:--|
| A | 語音辨識 (ASR) | Whisper | log-mel + 文字標籤 |
| B | 音訊分類 | wav2vec2 | 波形 + 類別標籤 |

音訊微調與文字/影像共用 `Trainer`，差別只在模型與前處理器。